# S3 Connectivity Exploration

Quick sanity check that the landing bucket is reachable and holds the expected files before wiring up the Snowflake storage integration and stage.

AWS credentials come from the default boto3 credential chain (AWS CLI profile / environment), never hardcoded here. Only the bucket name and prefix are read from `.env`.

In [ ]:
import os

import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

S3_BUCKET = os.environ["S3_BUCKET"]
S3_PREFIX = os.environ["S3_PREFIX"]

s3 = boto3.client("s3")

In [ ]:
# List the source files under the landing prefix.
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_PREFIX)

objects = response.get("Contents", [])
for obj in objects:
    print(f"{obj['Key']:60s} {obj['Size']:>10,d} bytes")

print(f"\n{len(objects)} file(s) found under s3://{S3_BUCKET}/{S3_PREFIX}")

In [ ]:
# Preview one file to confirm structure/encoding before it's loaded into Snowflake RAW.
sample_key = next(o["Key"] for o in objects if o["Key"].lower().endswith(".csv"))

obj = s3.get_object(Bucket=S3_BUCKET, Key=sample_key)
df_preview = pd.read_csv(obj["Body"], nrows=20)

print(f"Previewing: {sample_key}")
print(f"Columns: {list(df_preview.columns)}")
df_preview.head()